In [7]:
from datasets import load_dataset, concatenate_datasets

In [16]:
ds = load_dataset("sentence-transformers/codesearchnet")

print(len(ds["train"]))

1375067


In [17]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['comment', 'code'],
        num_rows: 1375067
    })
})


In [29]:
from pygments import lex
from pygments.lexers import guess_lexer_for_filename
from pygments.token import Token

EXTENSIONS = [".py", ".java", ".js", ".php", ".go", ".rb"]

def classify_code(example):
    code = example["code"]

    tokens = []
    classes = []

    lexer = None

    # try multiple extensions until one works well
    for ext in EXTENSIONS:
        try:
            candidate = guess_lexer_for_filename("file" + ext, code)
            # quick sanity check: must produce multi-char tokens
            sample = list(lex(code, candidate))[:10]
            if any(len(t[1].strip()) > 1 for t in sample):
                lexer = candidate
                break
        except:
            continue

    if lexer is None:
        return {"tokens": [], "classes": []}

    for tok_type, tok_val in lex(code, lexer):
        tok_val = tok_val.strip()
        if not tok_val:
            continue

        tokens.append(tok_val)

        if tok_type in Token.Keyword:
            classes.append("K")
        elif tok_type in Token.Name:
            classes.append("I")
        elif tok_type in Token.Literal:
            classes.append("L")
        elif tok_type in Token.Operator:
            classes.append("O")
        elif tok_type in Token.Punctuation:
            classes.append("P")
        else:
            classes.append("I")

    return {"tokens": tokens, "classes": classes}

In [30]:
small_ds = ds["train"].select(range(500))

In [31]:
small_ds = small_ds.map(classify_code)

Map: 100%|██████████| 500/500 [00:02<00:00, 246.80 examples/s]


In [32]:
print(small_ds.features)

{'comment': Value('string'), 'code': Value('string'), 'tokens': List(Value('string')), 'classes': List(Value('string'))}


In [33]:
sample = small_ds[0]

print(sample["tokens"][:20])
print(sample["classes"][:20])

['protected', 'function', 'parentId', '(', ')', '{', 'switch', '(', '$', 'this', '-', '>', 'position', ')', '{', 'case', "'", 'root', "'", ':']
['I', 'I', 'I', 'P', 'P', 'P', 'I', 'P', 'I', 'I', 'O', 'O', 'I', 'P', 'P', 'K', 'L', 'L', 'L', 'P']
